# kd-capacity-gap — v2 Experiment Runner (Colab Pro+ / A100)

End-to-end pipeline for the v2 evaluation protocol. **Run cells top-to-bottom.**
Every stage skips completed work (via `results.json`), and a background thread
syncs metrics + teacher checkpoints to Drive every 10 minutes — so a session
disconnect costs you at most the currently-running experiment. After a
disconnect, just reconnect an A100 runtime and re-run from the top.

| Stage | What happens | Est. time (A100) |
|-------|-------------|------------------|
| 1 | Teachers: R50, R34, R101 × 3 seeds (200 ep) | ~13–16 h |
| 2 | Selection grid: 4 pairs × 12 configs, seed 0 (val-based) | ~24–30 h |
| 3 | `collect_results --write-best` → `best_configs.json` | instant |
| 4 | Finals: baselines + best configs × 5 seeds + fidelity | ~28–34 h |
| 5 | Stem ablation | ~8–10 h |
| 6 | Aggregate + download metrics (small zip) | 1 min |

> **Total ≈ 75–90 A100-hours → expect 3–5 Colab sessions.** Enable
> **background execution** (Pro+ feature) so runs continue with the browser closed.
> Stages 2 and 4 will not fit in a single session — that's expected; resume works.
>
> ⚠ Student checkpoints are **not** synced to Drive (too large). If a session
> dies between a run finishing and its fidelity eval, that run's `fidelity.json`
> will be missing — Stage 4's last cell detects and backfills these.

## 0 — GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "NO GPU"
# If "NO GPU": Runtime -> Change runtime type -> A100 GPU, then re-run.

## 0.5 — Drive mount + background sync

`DRIVE_BASE` controls where outputs persist. The sync loop copies only:
- every `results.json`, `fidelity.json`, `best_configs.json`, `*.csv`
- teacher checkpoints (`checkpoints/teacher_*.pth` and `runs/teachers/**/checkpoint_best.pth`)

Student checkpoints stay local (disposable — fidelity is computed in-session).

In [ ]:
from google.colab import drive
import os, time, shutil, threading
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/kd-capacity-gap-v2')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')

REPO = Path('/content/kd-capacity-gap')

SYNC_PATTERNS = ['results.json', 'fidelity.json']

def _iter_sync_files():
    if not REPO.exists():
        return
    for pat in SYNC_PATTERNS:
        yield from (REPO / 'runs').rglob(pat) if (REPO / 'runs').exists() else []
    for extra in ['best_configs.json', 'final_results.csv']:
        p = REPO / extra
        if p.exists():
            yield p
    ck = REPO / 'checkpoints'
    if ck.exists():
        yield from ck.glob('teacher_*.pth')
    tr = REPO / 'runs' / 'teachers'
    if tr.exists():
        yield from tr.rglob('checkpoint_best.pth')

def sync_to_drive(verbose=False):
    n = 0
    for src in _iter_sync_files():
        rel = src.relative_to(REPO)
        dst = DRIVE_BASE / rel
        if not dst.exists() or dst.stat().st_mtime < src.stat().st_mtime:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    if verbose or n:
        print(f'[sync] {n} file(s) → Drive  ({time.strftime("%H:%M:%S")})')

def restore_from_drive():
    n = 0
    for src in DRIVE_BASE.rglob('*'):
        if not src.is_file():
            continue
        dst = REPO / src.relative_to(DRIVE_BASE)
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    print(f'[restore] {n} file(s) ← Drive')

def _sync_loop():
    while True:
        time.sleep(600)  # every 10 min
        try:
            sync_to_drive()
        except Exception as e:
            print(f'[sync] error: {e}')

if not any(t.name == 'drive-sync' for t in threading.enumerate()):
    threading.Thread(target=_sync_loop, name='drive-sync', daemon=True).start()
    print('Background sync started (every 10 min).')

## 1 — Clone repo, restore state, install deps

In [ ]:
import subprocess

REPO_URL = 'https://github.com/umutonuryasar/kd-capacity-gap.git'

if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('Working directory:', os.getcwd())

restore_from_drive()   # brings back results.json + teacher ckpts -> completed work is skipped


In [ ]:
# torch/torchvision are pre-installed on Colab
!pip install -r requirements.txt -q
print('Requirements ready.')
# CIFAR-10 downloads automatically to data/ on the first train.py call — no manual prep.

## 2 — Stage 1a: Teachers (R50, R34, R101 × 3 seeds, 200 ep)

`tools/train_teachers.sh` skips any (arch, seed) whose `results.json` exists.
Canonical checkpoints (`checkpoints/teacher_*.pth`) are the seed-0 best-val weights.

In [ ]:
!bash tools/train_teachers.sh
sync_to_drive(verbose=True)

In [ ]:
# Gate: verify teacher quality before spending grid compute.
# Expect val acc ≳ 95% for all three (may sit 0.1–0.3 pp below v1 — 45k train set).
!python tools/collect_results.py runs/teachers

## 3 — Stage 1b: Selection grid (48 runs, seed 0, val-based)

4 pairs × (9 Logit-KD + 3 Feature-KD) configs. Test set plays no role here.
Safe to interrupt: re-running the cell resumes where it left off.

In [ ]:
!bash tools/run_ablation.sh
sync_to_drive(verbose=True)

## 4 — Stage 1c: Select best configs by val accuracy

In [ ]:
!python tools/collect_results.py runs/select --write-best best_configs.json
sync_to_drive(verbose=True)

## 5 — Stage 2: Final runs (5 seeds) + fidelity

Baselines (R18, R34 × 5 seeds) + every best config × 5 seeds.
`tools/eval.py` runs automatically after each KD run (agreement, KL, per-class acc).
These are the numbers that go in the paper: `test_acc_best`, mean ± std.

In [ ]:
!bash tools/run_final.sh
sync_to_drive(verbose=True)

In [ ]:
# Backfill any fidelity.json lost to a session death between train and eval.
import json, subprocess
from pathlib import Path

best = json.load(open('best_configs.json'))
ckpt_for = {'resnet50': 'checkpoints/teacher_r50.pth',
            'resnet34': 'checkpoints/teacher_r34.pth',
            'resnet101': 'checkpoints/teacher_r101.pth'}
missing = 0
for name, b in best.items():
    for seed in range(5):
        out = Path(f"runs/final/{name.replace('/', '_')}/seed{seed}")
        if (out / 'results.json').exists() and not (out / 'fidelity.json').exists():
            student_ckpt = out / 'checkpoint_best.pth'
            if not student_ckpt.exists():
                print(f'  {out}: fidelity missing AND checkpoint gone — re-run this seed '
                      f'(delete its results.json, then re-run Stage 2).')
                missing += 1
                continue
            print(f'  Backfilling fidelity: {out}')
            subprocess.run(['python', 'tools/eval.py',
                            '--student-arch', b['student'], '--student-weights', str(student_ckpt),
                            '--teacher-arch', b['teacher'], '--teacher-weights', ckpt_for[b['teacher']],
                            '--output', str(out / 'fidelity.json')], check=True)
print('Fidelity check complete.' + (f'  ({missing} unrecoverable — see above)' if missing else ''))
sync_to_drive()

## 6 — Stage 3: Stem ablation (ImageNet vs CIFAR stem)

In [ ]:
!bash tools/run_stem_ablation.sh
sync_to_drive(verbose=True)
!python tools/collect_results.py runs/stem_ablation

## 7 — Aggregate + download

Zips **metrics only** (`results.json`, `fidelity.json`, CSV, `best_configs.json`) —
a few MB, not the multi-GB checkpoint archive. Upload this zip to the chat for
Phase 3 (paper revision).

In [ ]:
!python tools/collect_results.py runs/final --csv final_results.csv

In [ ]:
import subprocess
from pathlib import Path

files = [str(p) for p in Path('runs').rglob('results.json')]
files += [str(p) for p in Path('runs').rglob('fidelity.json')]
files += [f for f in ['final_results.csv', 'best_configs.json'] if Path(f).exists()]

archive = '/content/kd_v2_results.zip'
subprocess.run(['zip', '-q', archive] + files, check=True)
print(f'{archive}: {Path(archive).stat().st_size/1e6:.1f} MB, {len(files)} files')

shutil.copy2(archive, DRIVE_BASE / 'kd_v2_results.zip')
print('Also saved to Drive.')

from google.colab import files as colab_files
colab_files.download(archive)

---
### Session-resume cheat sheet

1. New runtime → **run cells 0 through 1** (mount, clone, restore, install).
2. Jump directly to the stage cell that was interrupted and re-run it.
3. Everything already completed is skipped automatically.

### Optional: feat_norm ablation
After Stage 1b, in a spare session: `!FEAT_NORM=teacher_std bash tools/run_ablation.sh`
run from a **separate clone** (it writes to the same `runs/select` layout) — ask
before running this one so we set up a clean output dir.